In [ ]:
# pratique_ibge_bcb.ipynb

import requests
import pandas as pd
from pathlib import Path

# -------------------------------
# 0. Criar pasta raw no caminho certo
# -------------------------------
OUTDIR = Path(r"------------------")
OUTDIR.mkdir(parents=True, exist_ok=True)

# -------------------------------
# 1. Consulta à API do IBGE
# -------------------------------
# Endpoint: lista de municípios
url_ibge = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"
resp_ibge = requests.get(url_ibge)
data_ibge = resp_ibge.json()

df_ibge = pd.DataFrame(data_ibge)
print("IBGE - Municípios (primeiras linhas):")
print(df_ibge.head())

# -------------------------------
# 2. Consulta ao Banco Central
# -------------------------------
# Endpoint: série histórica IPCA (SGS 433)
url_bcb = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json"
resp_bcb = requests.get(url_bcb)
data_bcb = resp_bcb.json()

df_bcb = pd.DataFrame(data_bcb)
print("Banco Central - IPCA (primeiras linhas):")
print(df_bcb.head())

# Salvar tabela bruta na pasta raw
df_bcb.to_csv(OUTDIR / "bcb_tabela.csv", index=False)

# -------------------------------
# 3. Tratamentos nos dados
# -------------------------------
# Ajuste de tipos
df_bcb["data"] = pd.to_datetime(df_bcb["data"], dayfirst=True)
df_bcb["valor"] = pd.to_numeric(df_bcb["valor"], errors="coerce")

# Renomeação de colunas
df_bcb.rename(columns={"data": "Data", "valor": "IPCA"}, inplace=True)

# Remoção de valores ausentes
df_bcb.dropna(inplace=True)

# Evidências
print("Tipos de dados:")
print(df_bcb.dtypes)
print("Valores nulos:", df_bcb.isnull().sum())

# -------------------------------
# 4. Salvar dados tratados na pasta raw
# -------------------------------
df_bcb.to_csv(OUTDIR / "dados_tratados.csv", index=False)

# Para parquet, lembre-se de instalar pyarrow ou fastparquet
try:
    df_bcb.to_parquet(OUTDIR / "dados_tratados.parquet", index=False)
except ImportError:
    print("⚠️ Instale 'pyarrow' ou 'fastparquet' para salvar em formato Parquet.")

print("Arquivos salvos em:", OUTDIR)
print(list(OUTDIR.glob("*")))